In [ ]:
#| export
# Lazy type-hint evaluation so `Path` doesn't need to be imported
# just to put it in a function signature.
from __future__ import annotations

In [ ]:
#| default_exp daily_summary


# Step 10: Daily digest

![Step 10 diagram](https://raw.githubusercontent.com/jaewilson07/bird-watcher/main/docs/diagrams/10-step.png)

**Goal:** once a day, post a summary to Slack — *'today we saw 7 birds, 4 unique species, top visitor was the American Robin.'*

We use the `schedule` library for the daily tick. In production you'd use a real cron, but `schedule` is fine for learning.

Three functions live in `bird_watcher/daily_summary.py`:

- `build_daily_summary` — build the Slack Block Kit payload
- `send_daily_summary` — build and send (or dry-run)
- `schedule_daily_summary` — blocking scheduler that posts once a day

## Step 10.0 — Setup

In [ ]:
from pathlib import Path

import yaml

ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next(
    (root for root in ROOT_CANDIDATES if (root / "tutorials").exists()),
    Path.cwd(),
)
CONFIG_FILE = PROJECT_ROOT / "config.yaml"

CONFIG = {}
if CONFIG_FILE.exists():
    CONFIG = yaml.safe_load(CONFIG_FILE.read_text()) or {}
    if not isinstance(CONFIG, dict):
        raise TypeError(f"{CONFIG_FILE} must contain a top-level mapping")
else:
    print(f"Config file not found at {CONFIG_FILE}; using defaults.")

# Folder + file constants. _FOLDER = a directory, _FILE = a single file.
TUTORIALS_FOLDER = PROJECT_ROOT / "tutorials"
DATA_FOLDER = PROJECT_ROOT / "data"
SNAPSHOT_FOLDER = DATA_FOLDER / "snapshots"
CROP_FOLDER = DATA_FOLDER / "crops"
DB_FILE = DATA_FOLDER / "birds.db"
SAMPLE_BIRD_FILE = DATA_FOLDER / "samples" / "sample-bird.jpg"
MODEL_FILE = PROJECT_ROOT / "yolov8n.pt"

# From config.yaml (or hardcoded fallback)
PHONE_IP = str(CONFIG.get("phone_ip", "192.168.1.42"))
PHONE_URL = f"http://{PHONE_IP}:8080/photo.jpg"
SLACK_WEBHOOK = str(CONFIG.get("slack_webhook", ""))
HUGGINGFACE_API_KEY = str(CONFIG.get("huggingface_api_key", ""))

SNAPSHOT_FOLDER.mkdir(parents=True, exist_ok=True)
CROP_FOLDER.mkdir(parents=True, exist_ok=True)
print(f"Snapshot folder: {SNAPSHOT_FOLDER}")
print(f"Config file: {CONFIG_FILE}")
print(f"Phone URL: {PHONE_URL}")


## Step 10.1 — Query the last 24 hours

`list_sightings_from_db` takes a `since=` timestamp. Compute it from `datetime.now()`.

In [ ]:
from datetime import datetime, timedelta
from bird_watcher.save_sighting import list_sightings_from_db

cutoff = (datetime.now() - timedelta(hours=24)).isoformat(timespec="seconds")
recent = list_sightings_from_db(DB_FILE, since=cutoff, limit=500, verbose=False)
print(f"{len(recent)} sighting(s) in the last 24h")

## Step 10.2 — Group by species, find the top visitor

In [ ]:
from collections import Counter

counts = Counter(s["species"] for s in recent)
print(f"{len(counts)} unique species")
for species, count in counts.most_common(5):
    print(f"  {species}: {count}")

## Step 10.3 — Wrap as `build_daily_summary`

In [ ]:
#| export
from collections import Counter
from datetime import datetime, timedelta
from pathlib import Path
from bird_watcher.save_sighting import list_sightings_from_db

def build_daily_summary(
    db_file: Path,
    snapshot_folder: Path,
    window_hours: int = 24,
) -> dict:
    """Build a Slack message summarizing the last `window_hours` of sightings.

    Args:
        db_file: where sightings are stored.
        snapshot_folder: where snapshot images live (for referencing).
        window_hours: how far back to look. Default 24 (one day).

    Returns:
        A Slack Block Kit payload.
    """
    from collections import Counter
    from datetime import datetime, timedelta

    cutoff = (datetime.now() - timedelta(hours=window_hours)).isoformat(timespec="seconds")
    sightings = list_sightings_from_db(db_file, since=cutoff, limit=500, verbose=False)

    total = len(sightings)
    counts = Counter(s["species"] for s in sightings)
    unique_species = len(counts)
    top_visitor = counts.most_common(1)[0] if counts else None

    title = f":bird: Daily summary — {total} sighting(s), {unique_species} species"
    if top_visitor:
        title += f", top: *{top_visitor[0]}*"

    blocks: list[dict] = [
        {"type": "header", "text": {"type": "plain_text", "text": title}},
        {
            "type": "section",
            "fields": [
                {"type": "mrkdwn", "text": f"*Window*\nlast {window_hours}h"},
                {"type": "mrkdwn", "text": f"*Total sightings*\n{total}"},
                {"type": "mrkdwn", "text": f"*Unique species*\n{unique_species}"},
            ],
        },
    ]
    if counts:
        species_lines = [
            f"• {species} — {count}" for species, count in counts.most_common(10)
        ]
        blocks.append(
            {
                "type": "section",
                "text": {"type": "mrkdwn", "text": "*Top species*\n" + "\n".join(species_lines)},
            }
        )

    return {"blocks": blocks}


## Step 10.4 — Add `send_daily_summary`

In [ ]:
#| export
from pathlib import Path
from bird_watcher.send_alert import send_alert_to_slack

def send_daily_summary(
    db_file: Path,
    snapshot_folder: Path,
    window_hours: int = 24,
    webhook_url: str | None = None,
    verbose: bool = True,
) -> bool:
    """Build and send (or dry-run) today's daily summary.

    Args:
        db_file: where sightings are stored.
        snapshot_folder: where snapshot images live.
        window_hours: how far back to look.
        webhook_url: optional Slack webhook URL. If omitted, this becomes a dry run.
        verbose: if True, print a status line.

    Returns:
        True if sent (or dry-run previewed), False on error.
    """
    from bird_watcher.send_alert import send_alert_to_slack

    payload = build_daily_summary(db_file, snapshot_folder, window_hours=window_hours)
    return send_alert_to_slack(payload, webhook_url=webhook_url, verbose=verbose)


## Step 10.5 — And the scheduler `schedule_daily_summary`

In [ ]:
#| export
import time
from pathlib import Path
import schedule

def schedule_daily_summary(
    db_file: Path,
    snapshot_folder: Path,
    run_at_hour: int = 21,
    verbose: bool = True,
) -> None:
    """Run a blocking scheduler that posts the daily summary once a day.

    For learning purposes only — production code would use a real cron job.

    Args:
        db_file: where sightings are stored.
        snapshot_folder: where snapshot images live.
        run_at_hour: hour of day to send (24h format). Default 21 (9pm).
        verbose: if True, print the next scheduled run.
    """
    import time

    import schedule

    job = lambda: send_daily_summary(db_file, snapshot_folder, verbose=verbose)
    schedule.every().day.at(f"{run_at_hour:02d}:00").do(job)
    if verbose:
        print(f"Scheduled daily summary at {run_at_hour:02d}:00 every day")
        print("Press Stop / Ctrl+C to exit")
    while True:
        schedule.run_pending()
        time.sleep(60)


## Acceptance criterion

In [ ]:
from bird_watcher.daily_summary import build_daily_summary, send_daily_summary

payload = build_daily_summary(DB_FILE, SNAPSHOT_FOLDER)
assert "blocks" in payload
assert any(b.get("type") == "header" for b in payload["blocks"])

sent = send_daily_summary(DB_FILE, SNAPSHOT_FOLDER)
assert sent, "send_daily_summary should return True (dry-run if no webhook)"
print("✅ Daily summary built + sent (or dry-run)")

## 🎉 You finished the bird watcher tutorial!

Ten notebooks. Seven modules. One bird watcher.

- pull photos from a phone camera
- find birds in those photos
- name the species
- log everything to a database
- post alerts to Slack
- serve a web gallery
- send a daily digest

**What's next?** Experiment. Add features. Break stuff. That's how you learn.